# aether — голосовая модель и ограниченная адаптация
Полный ручной сценарий: Git → окружение → данные → звук и текст → baseline loss →
пилот 5 шагов → resume до 20 шагов → сравнение → сохранённый адаптер.

Выберите **удалённый управляемый платный Colab, A100 40 ГБ или больше** для всего
сценария. L4 24 ГБ предназначена только для inference. T4 15 ГБ из предыдущего
отчёта не поддерживается этим BF16-профилем: отказ произойдёт до загрузки весов.
Это пороги допуска, не обещание отсутствия OOM. При OOM уменьшите frames до 32
или используйте больше памяти; checkpoint предыдущего шага сохранится.

Бюджет стадии — 500 юнитов. Notebook не умеет списывать/проверять их через API:
следите за расходом в интерфейсе. Обучение включается явно RUN_TRAINING=True.
Данные — короткие английские записи чтения: адаптация аудиогенерации, не обучение
содержательному диалогу. Прослушивание до/после обязательно для оценки результата.


In [ ]:
import json
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

CONFIRM_REMOTE_PAID_COLAB = False
RUN_TRAINING = False
RUN_ID = "english-demo-" + datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
DRIVE_ROOT = "/content/drive/MyDrive/aether"
REPO_URL = "https://github.com/karl4th/aether-v2.git"
GIT_REF = "main"  # For resume use source_revision from the saved run.json.
UV_VERSION = "0.12.13"
BUDGET_UNITS = 500
RESUME_CHECKPOINT = ""  # Optional COMPLETE checkpoint directory on Drive.
UPLOAD_QUESTION = False  # True: upload your English question as WAV/FLAC.

## Среда и точный checkout
Подтвердите выбор удалённой платной среды. Наличие библиотеки в изолированном uv
окружении не требуется. Для приватного GitHub используйте read-only Secret
GITHUB_TOKEN. Токен не сохраняется в URL или конфигурации Git.


In [ ]:
if not CONFIRM_REMOTE_PAID_COLAB:
    raise RuntimeError("Confirm managed remote paid Colab; local runtime is forbidden")
__import__("google.colab")
if sys.platform != "linux" or not Path("/content").is_dir():
    raise RuntimeError("Select a remote Colab Linux GPU runtime")

In [ ]:
import base64
import os
import tempfile

from google.colab import userdata


def checkout_source(repo_url, git_ref, workspace, git_env=None):
    if not git_ref or git_ref.startswith("-"):
        raise ValueError("Expected a Git branch, tag or commit SHA")
    project = Path(tempfile.mkdtemp(prefix="aether-src-", dir=workspace))
    env = dict(os.environ if git_env is None else git_env)
    env["GIT_TERMINAL_PROMPT"] = "0"

    def git(*args):
        result = subprocess.run(
            ["git", *args],
            cwd=project,
            env=env,
            check=True,
            capture_output=True,
            text=True,
            timeout=180,
        )
        return result.stdout.strip()

    git("init", "--quiet")
    git("remote", "add", "origin", repo_url)
    git("fetch", "--depth=1", "origin", git_ref)
    revision = git("rev-parse", "FETCH_HEAD^{commit}")
    git("checkout", "--detach", revision)
    for name in ("pyproject.toml", "uv.lock", ".python-version"):
        if not (project / name).is_file():
            raise ValueError("Missing required project file: " + name)
    return project, revision


# Optional read-only GitHub credential; never printed or stored in Git config.
git_env = os.environ.copy()
try:
    token = userdata.get("GITHUB_TOKEN")
except userdata.SecretNotFoundError:
    token = None
if token:
    if REPO_URL != "https://github.com/karl4th/aether-v2.git":
        raise ValueError("Credential use is restricted to the aether repository")
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    git_env.update(
        {
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": "Authorization: Basic " + auth,
        }
    )
    del auth
try:
    PROJECT, SOURCE_REVISION = checkout_source(REPO_URL, GIT_REF, "/content", git_env)
finally:
    git_env.clear()
    del token
print("Source commit:", SOURCE_REVISION)
print("Project:", PROJECT)

## Установка только в удалённой среде
uv использует lock-файл, Python 3.12.14 и отдельную .venv. Веса загрузятся позже,
после проверки ресурса. Каждый модельный вызов — отдельный процесс: память GPU
освобождается между baseline, обучением и оценкой.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==" + UV_VERSION], check=True)
UV = [sys.executable, "-m", "uv"]
subprocess.run(
    UV + ["sync", "--locked", "--no-dev", "--group", "model", "--python", "3.12.14"],
    cwd=PROJECT,
    check=True,
)


def package_run(*arguments):
    result = subprocess.run(
        UV + ["run", "--locked", "--no-dev", "--group", "model", *map(str, arguments)],
        cwd=PROJECT,
        text=True,
        capture_output=True,
    )
    if result.returncode:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("aether operation failed; inspect the diagnostic above")
    return result


def command_json(*arguments):
    result = package_run("aether", *arguments)
    if result.stderr:
        print(result.stderr[-4000:])
    return json.loads(result.stdout.splitlines()[-1])


print(
    package_run(
        "aether", "train", "--config", "configs/training/colab_lora.json", "--validate-only"
    ).stdout
)

## Разрешение текущему runtime и хранилище
Разрешение привязано к boot ID и живому процессу notebook, действует до 12 часов.
Это защита от случайного локального запуска, не криптографическое доказательство
тарифа. Явное подтверждение платного runtime дополняется проверкой нескольких
признаков среды. Результаты и checkpoint записываются в новый run на Google Drive.


In [ ]:
import hashlib
import os
import time

from google.colab import drive

drive.mount("/content/drive")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive is not mounted")
PERMIT = PROJECT / "runtime-permit.json"
now = time.time()
PERMIT.write_text(
    json.dumps(
        {
            "schema_version": 1,
            "boot_id": Path("/proc/sys/kernel/random/boot_id").read_text().strip(),
            "kernel_pid": os.getpid(),
            "created_at": now,
            "expires_at": now + 12 * 3600,
            "user_confirmed_remote_paid": CONFIRM_REMOTE_PAID_COLAB,
            "allow_training": RUN_TRAINING,
            "budget_units": BUDGET_UNITS,
            "source_revision": SOURCE_REVISION,
        }
    )
)
result = package_run(
    "python",
    "-c",
    "import sys; from pathlib import Path; from aether.storage import create_run; "
    "print(create_run(Path(sys.argv[1]),sys.argv[2]))",
    DRIVE_ROOT,
    RUN_ID,
)
RUN_DIR = Path(result.stdout.strip())
report = json.loads(
    package_run(
        "python",
        "-c",
        "import json; from aether.preflight import collect_preflight; "
        "print(json.dumps(collect_preflight()))",
    ).stdout
)
report.update(
    source_revision=SOURCE_REVISION,
    source_repository=REPO_URL,
    uv_lock_sha256=hashlib.sha256((PROJECT / "uv.lock").read_bytes()).hexdigest(),
    budget_units=BUDGET_UNITS,
    training_requested=RUN_TRAINING,
)
(RUN_DIR / "run.json").write_text(json.dumps(report, indent=2))
print(report["gpu"])
package_run(
    "python",
    "-c",
    "import sys; from aether.remote import require_remote_runtime; "
    "require_remote_runtime(permit_path=sys.argv[1]); import torch; "
    "from aether.backend import check_resources; "
    "check_resources(torch, training=sys.argv[2]=='True')",
    PERMIT,
    str(RUN_TRAINING),
)
print("Resource admission passed; actual peak memory is checked by real operations below.")

## Английские данные
Загружаются официальные архивы небольшого корпуса с проверкой опубликованных
checksum. Выбираются 16 train и 4 evaluation записей 2–5 секунд; speakers не
пересекаются. Исходное аудио и условия CC BY 4.0 сохраняются. На Drive копируются
манифест и атрибуция, рабочее аудио остаётся на диске VM и воспроизводимо загружается
снова для resume. SHA256 выбранных аудиофайлов входит в идентичность датасета.


In [ ]:
import shutil

DATA_ROOT = Path("/content/aether-data")
data_result = command_json(
    "prepare-data",
    "--output",
    DATA_ROOT,
    "--permit",
    PERMIT,
    "--max-train",
    "16",
    "--max-eval",
    "4",
    "--max-seconds",
    "5",
)
DATASET = Path(data_result["manifest"])
shutil.copy2(DATASET, RUN_DIR / "dataset.json")
shutil.copy2(DATA_ROOT / "ATTRIBUTION.txt", RUN_DIR / "DATA_ATTRIBUTION.txt")
raw_dataset = json.loads(DATASET.read_text())
INPUT_AUDIO = DATA_ROOT / raw_dataset["evaluation"][0]["audio_path"]
if UPLOAD_QUESTION:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload one short English question as WAV or FLAC")
    INPUT_AUDIO = PROJECT / "question.wav"
    INPUT_AUDIO.write_bytes(next(iter(uploaded.values())))
print(data_result)

## Inference с прослушиванием
Фиксированная ревизия согласованного комплекта: голосовая модель, начальный кодек,
токенизатор. При первой операции загрузится около 16 ГБ; последующие используют
локальный кеш. Сохраняются хеши весов, seed, ревизии и параметры. Без upload вход —
проверочная запись чтения; для демо вопрос/ответ включите UPLOAD_QUESTION выше.


In [ ]:
from IPython.display import Audio, display

baseline = command_json(
    "infer",
    "--config",
    "configs/model/remote.json",
    "--permit",
    PERMIT,
    "--input",
    INPUT_AUDIO,
    "--output",
    RUN_DIR / "baseline",
)
print("Input:")
display(Audio(filename=str(INPUT_AUDIO)))
print(baseline["text"])
display(Audio(filename=baseline["audio"]))

## Оценка до адаптации
Teacher-forced audio cross-entropy/perplexity по отложенным speakers. Текст без
word timestamps исключается из loss: он не получает выдуманного выравнивания.
Эта метрика не заменяет оценку качества диалога или разборчивости.


In [ ]:
before = command_json(
    "evaluate",
    "--config",
    "configs/training/colab_lora.json",
    "--permit",
    PERMIT,
    "--dataset",
    DATASET,
    "--output",
    RUN_DIR / "before.json",
)
print(before)

## Pilot → сохранение → новый процесс → resume
Только при RUN_TRAINING=True. Frozen base и начальный кодек, rank-8 адаптер
последнего temporal FFN, 64 кадра, batch=1, 20 шагов, learning rate 1e-4.
Пилот выполняет 5 шагов из общего плана, сохраняет optimizer/scheduler/RNG,
затем второй процесс восстанавливает checkpoint и продолжает до 20.
Обучающий цикл ограничен 30 минутами (загрузка не входит); при достижении лимита сохранится неполный результат.
Для новой сессии задайте RESUME_CHECKPOINT и исходный GIT_REF; параметры не меняйте.
Нельзя обещать 20 шагов за заданное время до измерения на выделенной GPU.


In [ ]:
FINAL_CHECKPOINT = None
if RUN_TRAINING:
    if RESUME_CHECKPOINT:
        checkpoint = RESUME_CHECKPOINT
    else:
        pilot = command_json(
            "train",
            "--config",
            "configs/training/colab_lora.json",
            "--permit",
            PERMIT,
            "--dataset",
            DATASET,
            "--output",
            RUN_DIR,
            "--stop-after-steps",
            "5",
        )
        checkpoint = pilot["checkpoint"]
        (RUN_DIR / "pilot.json").write_text(json.dumps(pilot, indent=2))
    trained = command_json(
        "train",
        "--config",
        "configs/training/colab_lora.json",
        "--permit",
        PERMIT,
        "--dataset",
        DATASET,
        "--output",
        RUN_DIR,
        "--resume",
        checkpoint,
    )
    FINAL_CHECKPOINT = trained["checkpoint"]
    (RUN_DIR / "training.json").write_text(json.dumps(trained, indent=2))
    print(
        {"completed": trained["completed"], "step": trained["step"], "checkpoint": FINAL_CHECKPOINT}
    )
else:
    print("Baseline completed. Training was not requested; no optimizer was constructed.")

## После адаптации: та же оценка и тот же звук
Checkpoint применяется к той же ревизии базовой модели; не объединяется с чужими
весами. Сравните loss и прослушайте результат. Уменьшение loss не доказывает
улучшение разговорного качества; ограниченный пилот служит проверкой контура.


In [ ]:
if FINAL_CHECKPOINT:
    after = command_json(
        "evaluate",
        "--config",
        "configs/training/colab_lora.json",
        "--permit",
        PERMIT,
        "--dataset",
        DATASET,
        "--checkpoint",
        FINAL_CHECKPOINT,
        "--output",
        RUN_DIR / "after.json",
    )
    candidate = command_json(
        "infer",
        "--config",
        "configs/model/remote.json",
        "--permit",
        PERMIT,
        "--input",
        INPUT_AUDIO,
        "--checkpoint",
        FINAL_CHECKPOINT,
        "--output",
        RUN_DIR / "adapted",
    )
    print({"before_audio_ce": before["audio_ce"], "after_audio_ce": after["audio_ce"]})
    print(candidate["text"])
    display(Audio(filename=candidate["audio"]))

## Экспорт и завершение
Экспорт содержит адаптер, optimizer/RNG для восстановления, манифест, хеши и
атрибуцию. Базовые веса повторно не копируются. Используйте ту же ревизию кода
для возобновления; при несовпадении identity загрузка отвергается.


In [ ]:
if FINAL_CHECKPOINT:
    export_dir = Path(DRIVE_ROOT) / "exports" / RUN_ID
    package_run(
        "python",
        "-c",
        "import sys; from pathlib import Path; from aether.storage import verify_checkpoint; "
        "verify_checkpoint(Path(sys.argv[1]))",
        FINAL_CHECKPOINT,
    )
    shutil.copytree(FINAL_CHECKPOINT, export_dir)
    package_run(
        "python",
        "-c",
        "import sys; from pathlib import Path; from aether.storage import verify_checkpoint; "
        "verify_checkpoint(Path(sys.argv[1]))",
        export_dir,
    )
    shutil.copy2(PROJECT / "THIRD_PARTY_NOTICES.md", RUN_DIR / "THIRD_PARTY_NOTICES.md")
    print("Export verified:", export_dir)
print("Results:", RUN_DIR)
print("Disconnect the GPU runtime when finished to stop idle resource consumption.")